In [1]:
!pip install autogluon.tabular

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.8/514.8 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.4/248.4 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.3/112.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.7/90.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import os
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [15]:
datasetPath = Path("/content/drive/MyDrive/ML-For-CV-Robustness/datasets/notrees.csv")

#making sure
datasetPath.is_file()

True

In [16]:
dataset = pd.read_csv(datasetPath)
dataset.head()

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
0,0.163549,0.000324,0.000421,1130.973355,1020.232214,0.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0,0.088270
1,0.097110,0.000412,0.000486,1130.973355,1020.232214,1.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0,0.057202
2,0.074345,0.000448,0.000517,1130.973355,1020.232214,2.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0,0.048503
3,0.156574,0.000374,0.000481,1130.973355,1020.232214,3.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0,0.086477
4,0.116627,0.000393,0.000479,1130.973355,1020.232214,4.0,180.0,-63.812729,-56.07412,0.0,-19.258,-0.0,0.067095


In [17]:
from autogluon.tabular import TabularPredictor

In [18]:
train_df = dataset.sample(frac=0.8, random_state=42)
test_df = dataset.drop(train_df.index)

In [19]:
train_df

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
142331,0.145998,0.000409,0.000513,531.714557,479.878278,11.0,180.0,-63.688171,56.363342,0.0,-19.532,-0.000,0.078595
270207,0.311861,0.000227,0.000388,368.744438,321.424198,7.0,180.0,-42.151062,85.965363,0.0,-19.320,-0.000,0.137009
49351,0.270059,0.000188,0.000296,220.893233,200.472881,11.0,180.0,-71.945084,19.489212,0.0,-22.271,180.000,0.093232
169549,0.229224,0.000165,0.000216,1130.973355,1020.232214,9.0,180.0,-63.812729,-56.074120,-0.0,-22.112,-180.000,0.086487
31301,0.304657,0.000192,0.000316,1334.391480,1020.232214,1.0,180.0,-19.763107,-104.398338,0.0,-22.510,180.000,0.129195
...,...,...,...,...,...,...,...,...,...,...,...,...,...
304611,0.233003,0.000227,0.000312,531.714557,479.878278,11.0,180.0,-63.688171,56.363342,-0.0,-21.426,-180.000,0.080990
119616,0.270809,0.000170,0.000217,370.118885,322.798645,16.0,180.0,-42.301559,-85.835487,0.0,-19.813,81.537,0.088975
59809,0.250882,0.000259,0.000396,228.550866,199.294784,9.0,180.0,-42.151062,85.965363,0.0,-18.992,-0.000,0.101696
273354,0.222158,0.000413,0.000600,368.744438,321.424198,14.0,180.0,-42.151062,85.965363,-0.0,-18.932,-0.000,0.094931


In [20]:
test_df

,grviIn,meanRefR,meanRefG,illumR,illumG,blockIdx,sunRoll,sunPitch,sunYaw,camRoll,camPitch,camYaw,grviOut
5,0.081775,0.000463,0.000538,1130.973355,1020.232214,5.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.053121
13,0.108716,0.000371,0.000444,1130.973355,1020.232214,13.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.066583
15,0.144419,0.000328,0.000411,1130.973355,1020.232214,15.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.087779
18,0.167532,0.000247,0.000317,1130.973355,1020.232214,18.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.094252
19,0.185488,0.000243,0.000321,1130.973355,1020.232214,19.0,180.0,-63.812729,-56.074120,0.0,-19.258,-0.0,0.097285
...,...,...,...,...,...,...,...,...,...,...,...,...,...
333458,0.296286,0.000274,0.000436,550.956812,480.270977,18.0,180.0,-42.301559,-85.835487,-0.0,-22.204,-180.0,0.105932
333460,0.267418,0.000102,0.000135,550.956812,480.270977,0.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.101249
333470,0.239545,0.000260,0.000367,550.956812,480.270977,10.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.084095
333473,0.259494,0.000291,0.000437,550.956812,480.270977,13.0,180.0,-42.301559,-85.835487,-0.0,-22.218,-180.0,0.085348


In [21]:
predictor = TabularPredictor(
    label="grviOut",
    problem_type="regression",
    eval_metric="root_mean_squared_error"
).fit(
    train_data=train_df
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260822_145624"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          2
Pytorch Version:    2.11.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB
Total GPU Memory:   Free: 14.56 GB, Allocated: 0.00 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       10.71 GB / 12.67 GB (84.5%)
Disk Space Avail:   10.57 GB / 15.00 GB (70.5%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and th

[1000]	valid_set's rmse: 0.00685107
[2000]	valid_set's rmse: 0.00631546
[3000]	valid_set's rmse: 0.00605433
[4000]	valid_set's rmse: 0.00589367
[5000]	valid_set's rmse: 0.0057869
[6000]	valid_set's rmse: 0.0057035
[7000]	valid_set's rmse: 0.00563611
[8000]	valid_set's rmse: 0.00558198
[9000]	valid_set's rmse: 0.00553672
[10000]	valid_set's rmse: 0.00549514


	-0.0055	 = Validation score   (-root_mean_squared_error)
	160.08s	 = Training   runtime
	2.89s	 = Validation runtime
Fitting model: LightGBM ...
	Fitting with cpus=1, gpus=0, mem=0.2/10.5 GB


[1000]	valid_set's rmse: 0.00620989
[2000]	valid_set's rmse: 0.0057276
[3000]	valid_set's rmse: 0.00551158
[4000]	valid_set's rmse: 0.00535588
[5000]	valid_set's rmse: 0.0052566
[6000]	valid_set's rmse: 0.00518657
[7000]	valid_set's rmse: 0.00512055
[8000]	valid_set's rmse: 0.00506935
[9000]	valid_set's rmse: 0.005027
[10000]	valid_set's rmse: 0.00499056


	-0.005	 = Validation score   (-root_mean_squared_error)
	111.61s	 = Training   runtime
	1.96s	 = Validation runtime
Fitting model: RandomForestMSE ...
	Fitting with cpus=2, gpus=0, mem=1.3/10.5 GB
	-0.0066	 = Validation score   (-root_mean_squared_error)
	561.03s	 = Training   runtime
	0.31s	 = Validation runtime
Fitting model: CatBoost ...
	Fitting with cpus=1, gpus=0
		`import catboost` failed. A quick tip is to install via `pip install autogluon.tabular[catboost]==1.6.1`.
Fitting model: ExtraTreesMSE ...
	Fitting with cpus=2, gpus=0, mem=1.3/10.5 GB
	-0.006	 = Validation score   (-root_mean_squared_error)
	109.63s	 = Training   runtime
	0.28s	 = Validation runtime
Fitting model: NeuralNetFastAI ...
	Fitting with cpus=1, gpus=0, mem=0.2/10.4 GB
	-0.0069	 = Validation score   (-root_mean_squared_error)
	191.17s	 = Training   runtime
	0.04s	 = Validation runtime
Fitting model: XGBoost ...
	Fitting with cpus=1, gpus=0
	-0.0049	 = Validation score   (-root_mean_squared_error)
	103.94s	 

[1000]	valid_set's rmse: 0.00557712
[2000]	valid_set's rmse: 0.00525023
[3000]	valid_set's rmse: 0.00507904
[4000]	valid_set's rmse: 0.00495934
[5000]	valid_set's rmse: 0.00487338
[6000]	valid_set's rmse: 0.00481768
[7000]	valid_set's rmse: 0.00476958
[8000]	valid_set's rmse: 0.00473911
[9000]	valid_set's rmse: 0.00471085
[10000]	valid_set's rmse: 0.00468842


	-0.0047	 = Validation score   (-root_mean_squared_error)
	131.91s	 = Training   runtime
	2.43s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ...
	Fitting 1 model on all data | Fitting with cpus=2, gpus=0, mem=0.0/10.0 GB
	Ensemble Weights: {'LightGBMLarge': 0.667, 'XGBoost': 0.267, 'ExtraTreesMSE': 0.067}
	-0.0046	 = Validation score   (-root_mean_squared_error)
	0.02s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 2092.59s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 896.1 rows/s (2668 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/content/drive/MyDrive/ML-For-CV-Robustness/datasets/AutogluonModels/ag-20260822_145624")


In [22]:
predictor.leaderboard()

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.004645,root_mean_squared_error,2.977369,345.498397,0.000458,0.019172,2,True,9
1,LightGBMLarge,-0.004688,root_mean_squared_error,2.434174,131.911205,2.434174,131.911205,1,True,8
2,XGBoost,-0.004898,root_mean_squared_error,0.267084,103.936441,0.267084,103.936441,1,True,6
3,LightGBM,-0.004991,root_mean_squared_error,1.955552,111.614114,1.955552,111.614114,1,True,2
4,LightGBMXT,-0.005495,root_mean_squared_error,2.886207,160.080127,2.886207,160.080127,1,True,1
5,NeuralNetTorch,-0.005710,root_mean_squared_error,0.024336,690.272964,0.024336,690.272964,1,True,7
6,ExtraTreesMSE,-0.005991,root_mean_squared_error,0.275653,109.631578,0.275653,109.631578,1,True,4
7,RandomForestMSE,-0.006560,root_mean_squared_error,0.312938,561.028113,0.312938,561.028113,1,True,3
8,NeuralNetFastAI,-0.006933,root_mean_squared_error,0.037596,191.167501,0.037596,191.167501,1,True,5


In [23]:
predictor.leaderboard(test_df, silent=True)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.004615,-0.004645,root_mean_squared_error,76.206679,2.977369,345.498397,0.030692,0.000458,0.019172,2,True,9
1,LightGBMLarge,-0.004646,-0.004688,root_mean_squared_error,64.027623,2.434174,131.911205,64.027623,2.434174,131.911205,1,True,8
2,LightGBM,-0.004864,-0.004991,root_mean_squared_error,48.230255,1.955552,111.614114,48.230255,1.955552,111.614114,1,True,2
3,XGBoost,-0.004922,-0.004898,root_mean_squared_error,6.711711,0.267084,103.936441,6.711711,0.267084,103.936441,1,True,6
4,LightGBMXT,-0.005369,-0.005495,root_mean_squared_error,70.662203,2.886207,160.080127,70.662203,2.886207,160.080127,1,True,1
5,NeuralNetTorch,-0.005769,-0.005710,root_mean_squared_error,0.354123,0.024336,690.272964,0.354123,0.024336,690.272964,1,True,7
6,ExtraTreesMSE,-0.005967,-0.005991,root_mean_squared_error,5.436652,0.275653,109.631578,5.436652,0.275653,109.631578,1,True,4
7,RandomForestMSE,-0.006677,-0.006560,root_mean_squared_error,3.876890,0.312938,561.028113,3.876890,0.312938,561.028113,1,True,3
8,NeuralNetFastAI,-0.006944,-0.006933,root_mean_squared_error,0.773612,0.037596,191.167501,0.773612,0.037596,191.167501,1,True,5
